In [ ]:
import pickle
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import TimeSeriesSplit
from sklearn.linear_model import Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_regression

In [ ]:
with open('data/datasets/data_cleaned.pkl', 'rb') as f:
    states_dfs = pickle.load(f)

The states that SARIMA does not work as well for

In [ ]:
states = ['OR', 'NY', 'NV', 'NH', 'ME', 'LA', 'HI', 'FL', 'CT']
target = 'residential_electricity_price'

In [ ]:
kfold = TimeSeriesSplit(n_splits=2, test_size=12)

In [ ]:
def plot_random_forrest_features(X: pd.DataFrame, y, state='HI', n_estimators=100, random_state=42):
    model = RandomForestRegressor(n_estimators=n_estimators, random_state=random_state)
    max_date = X.index.max()
    model.fit(X, y)
    feature_importance = model.feature_importances_
    importances = pd.Series(data=feature_importance, index=X.columns)
    importances.sort_values(inplace=True, ascending=False)
    plt.figure(figsize=(20, 7))
    plt.bar(importances.index, importances)
    plt.xlabel('feature')
    plt.ylabel('importance')
    plt.xticks(rotation=90, fontsize=14)
    title = f'Feature Importance for state: {state}. Training data max date: {max_date.strftime('%b %Y')}'
    plt.title(title, fontsize=16)
    plt.tight_layout()
    plt.show()

In [ ]:
def plot_lasso_coeffs(X: pd.DataFrame, y, state='HI', alpha=1e-1):
    model = Lasso(alpha=alpha, max_iter=10000)
    model.fit(X, y)
    max_date =X.index.max()
    coeffs = pd.Series(data=np.abs(model.coef_), index=X.columns)
    coeffs.sort_values(inplace=True, ascending=False)
    plt.figure(figsize=(20, 7))
    plt.bar(coeffs.index, coeffs)
    plt.xlabel('feature')
    plt.ylabel('importance')
    plt.xticks(rotation=90, fontsize=14)
    title = f'Absolute value of Lasso Coefficents for state: {state}. Training data max date: {max_date.strftime('%b %Y')}'
    plt.title(title, fontsize=16)
    plt.tight_layout()
    plt.show()



In [ ]:
def plot_select_k_best_scores(X: pd.DataFrame, y, state='HI', k=10):
    selector = SelectKBest(f_regression, k=k)
    selector.fit(X, y)
    max_date =X.index.max()
    scores = pd.Series(data=selector.scores_, index=X.columns)
    scores.sort_values(inplace=True, ascending=False)
    plt.figure(figsize=(20, 7))
    plt.bar(scores.index, scores)
    plt.xlabel('feature')
    plt.ylabel('Score')
    plt.xticks(rotation=90, fontsize=14)
    title = f'SelectKBest scores for state: {state}. Training data max date: {max_date.strftime('%b %Y')}'
    plt.title(title, fontsize=16)
    plt.tight_layout()
    plt.show()



In [ ]:
for state in states:
    df = states_dfs[state]
    df = df.dropna(axis=1)
    constant_filter = VarianceThreshold(threshold=0)
    constant_filter.fit(df)
    df = df.iloc[:, constant_filter.get_support()]
    print(f'STATE: {state}')
    for train_idx, _ in kfold.split(df):
        X = df.iloc[train_idx, :].drop(columns=[target])
        y = df.iloc[train_idx, :][target]
        plot_random_forrest_features(X, y, state=state)
        plot_lasso_coeffs(X, y, state=state)
        plot_select_k_best_scores(X, y, state=state)
    print('')

